# Training Ready Process

이 노트북은 `detail_pages_sanitized.csv` 를 입력으로 받아 실제 학습용 테이블을 생성합니다.

처리 순서는 다음과 같습니다.

- base row 정리
- `string2BMT`
- `BMTaddCAT`
- `BMT2Score`
- CSV export

CSV 는 한 번만 읽고, 이후에는 메모리 안의 row 목록을 stage 단위로 넘기면서 처리합니다. 각 stage 는 진행 로그와 성공/실패 집계를 출력합니다.


In [1]:
import csv
import sys
import time
from collections import Counter
from pathlib import Path
from pprint import pprint


## 1. 경로 설정

현재 작업 디렉터리와 상관없이 프로젝트 루트를 찾고 입력/출력 파일 경로를 고정합니다.


In [2]:
def find_project_root(start: Path) -> Path:
    for base in (start, *start.parents):
        candidate = base / 'data_collection' / 'clean' / 'output' / 'detail_pages_sanitized.csv'
        if candidate.exists():
            return base
    raise FileNotFoundError('Could not locate project root from the current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUT_DIR = PROJECT_ROOT / 'data_collection' / 'clean' / 'output'
SANITIZED_CSV = OUTPUT_DIR / 'detail_pages_sanitized.csv'
TRAINING_READY_CSV = OUTPUT_DIR / 'detail_pages_training_ready.csv'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'project_root: {PROJECT_ROOT}')
print(f'sanitized_csv: {SANITIZED_CSV}')
print(f'training_ready_csv: {TRAINING_READY_CSV}')


project_root: /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team
sanitized_csv: /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/clean/output/detail_pages_sanitized.csv
training_ready_csv: /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/clean/output/detail_pages_training_ready.csv


## 2. 모듈 import

세 패키지를 직접 import 합니다. `string2BMT` 는 현재 `pydantic` 의존성이 필요합니다.


In [3]:
try:
    from data_collection.name_brand_model_trim_mapping.string2BMT.normalizer import load_string_to_bmt_mapper
    from data_collection.brand_model_trim_category_mapping.BMTaddCAT import get_major_category_from_tuple
    from data_collection.cohort_score_calculator.BMT2Score import get_size_score
except (ModuleNotFoundError, ImportError) as exc:
    raise ImportError(
        'training_ready_process.ipynb requires importable mapping modules. '
        'Check string2BMT dependencies and restart the notebook kernel if package exports changed.'
    ) from exc

STRING_TO_BMT_MAPPER = load_string_to_bmt_mapper()

print('module import check: OK')


module import check: OK


## 3. Sanitize 결과 로드

입력 CSV 를 읽고 기본 행 수, 컬럼, title 중복 정도를 확인합니다. 이 중복 수는 뒤쪽 캐시 효과를 확인할 때 유용합니다.


In [4]:
with SANITIZED_CSV.open('r', encoding='utf-8-sig', newline='') as file:
    sanitized_rows = list(csv.DictReader(file))

if TRAINING_READY_CSV.exists():
    with TRAINING_READY_CSV.open('r', encoding='utf-8-sig', newline='') as file:
        existing_training_ready_rows = list(csv.DictReader(file))
else:
    existing_training_ready_rows = []

title_counter = Counter(row['title'] for row in sanitized_rows)

print(f'sanitized row count: {len(sanitized_rows):,}')
print(f'unique title count: {len(title_counter):,}')
print(f'existing training-ready cache rows: {len(existing_training_ready_rows):,}')
print('sanitized columns:')
pprint(list(sanitized_rows[0].keys()) if sanitized_rows else [])
print('\nmost common duplicate titles:')
pprint(title_counter.most_common(10))


sanitized row count: 2,468
unique title count: 1,038
existing training-ready cache rows: 2,084
sanitized columns:
['title',
 'registration_number',
 'price_manwon',
 'model_year',
 'first_registration_yyyymm',
 'color',
 'mileage_km']

most common duplicate titles:
[('[기아] 올 뉴카니발 9인승 디젤 프레스티지', 26),
 ('[기아] 신형 카니발(KA4) 9인승 디젤 노블레스', 24),
 ('[기아] 올 뉴모닝(JA) 가솔린 럭셔리', 23),
 ('[기아] 신형 카니발(KA4) 9인승 디젤 시그니처', 22),
 ('[쉐보레(대우)] 더 넥스트 스파크 LT 플러스', 20),
 ('[제네시스] 더 올뉴G80 가솔린 2.5 AWD 기본형', 20),
 ('[기아] 신형 카니발(KA4) 9인승 디젤 프레스티지', 19),
 ('[기아] 더 뉴레이 밴 스탠다드', 19),
 ('[기아] 더 뉴카니발(YP) 9인승 디젤 프레스티지', 16),
 ('[현대] 아반떼AD 1.6 GDi 밸류 플러스', 15)]


## 4. 파이프라인 설정과 stage 함수 정의

여기서는 row-by-row 처리를 감춘 하나의 `enrich_row` 대신, stage 단위 배치 파이프라인을 정의합니다. 각 stage 는 입력 row 목록을 받아 새 row 목록을 반환하고, 진행 로그를 출력합니다.


In [5]:
USE_AGENT = True
LOG_EVERY = 26
BMT_BATCH_SIZE = 25
BMT_MAX_CONCURRENCY = 25
BMT_MAX_RETRIES = 2
BMT_RETRY_BACKOFF_SECONDS = 1.0

OUTPUT_COLUMNS = [
    'title',
    'registration_number',
    'price_manwon',
    'model_year',
    'first_registration_yyyymm',
    'color',
    'mileage_km',
    'brand',
    'model_name',
    'trim_name',
    'major_category',
    'size_score',
]

bmt_cache: dict[str, tuple[str | None, str | None, str | None]] = {}
category_cache: dict[tuple[str | None, str | None, str | None], str | None] = {}
score_cache: dict[tuple[str | None, str | None, str | None], float | None] = {}


def normalize_optional_text(value: object) -> str | None:
    text = str(value or '').strip()
    return text or None


def parse_optional_int(value: str | None) -> int | None:
    text = (value or '').strip()
    if not text:
        return None
    return int(text)


def parse_optional_float(value: str | None) -> float | None:
    text = (value or '').strip()
    if not text:
        return None
    return float(text)


def log_stage_header(stage_name: str, total: int) -> float:
    started_at = time.perf_counter()
    print(f'[{stage_name}] start: total_rows={total:,}')
    return started_at


def log_stage_progress(stage_name: str, index: int, total: int, *, extra: str = '') -> None:
    if index % LOG_EVERY == 0 or index == total:
        suffix = f' | {extra}' if extra else ''
        print(f'[{stage_name}] progress: {index:,}/{total:,}{suffix}')


def log_stage_footer(stage_name: str, started_at: float, **metrics: object) -> None:
    elapsed = time.perf_counter() - started_at
    metric_text = ', '.join(f'{key}={value}' for key, value in metrics.items())
    print(f'[{stage_name}] done: elapsed={elapsed:.2f}s, {metric_text}')


def seed_caches_from_existing_rows(rows: list[dict[str, str]]) -> dict[str, int]:
    seeded_bmt_titles = 0
    seeded_categories = 0
    seeded_scores = 0

    for row in rows:
        title = normalize_optional_text(row.get('title'))
        brand = normalize_optional_text(row.get('brand'))
        model_name = normalize_optional_text(row.get('model_name'))
        trim_name = normalize_optional_text(row.get('trim_name'))
        major_category = normalize_optional_text(row.get('major_category'))
        size_score = parse_optional_float(row.get('size_score'))

        if title and title not in bmt_cache:
            bmt_cache[title] = (brand, model_name, trim_name)
            seeded_bmt_titles += 1

        if brand and model_name:
            tuple_key = (brand, model_name, trim_name)
            if major_category is not None and tuple_key not in category_cache:
                category_cache[tuple_key] = major_category
                seeded_categories += 1
            if size_score is not None and tuple_key not in score_cache:
                score_cache[tuple_key] = size_score
                seeded_scores += 1

    return {
        'seeded_bmt_titles': seeded_bmt_titles,
        'seeded_categories': seeded_categories,
        'seeded_scores': seeded_scores,
    }


seed_metrics = seed_caches_from_existing_rows(existing_training_ready_rows)
print(
    'warm cache loaded: '
    f"titles={seed_metrics['seeded_bmt_titles']:,}, "
    f"categories={seed_metrics['seeded_categories']:,}, "
    f"scores={seed_metrics['seeded_scores']:,}"
)


def build_base_rows(rows: list[dict[str, str]]) -> list[dict[str, object]]:
    stage_name = 'base'
    started_at = log_stage_header(stage_name, len(rows))
    built_rows: list[dict[str, object]] = []

    for index, row in enumerate(rows, start=1):
        built_rows.append(
            {
                'title': (row.get('title') or '').strip(),
                'registration_number': (row.get('registration_number') or '').strip(),
                'price_manwon': parse_optional_int(row.get('price_manwon')),
                'model_year': parse_optional_int(row.get('model_year')),
                'first_registration_yyyymm': parse_optional_int(row.get('first_registration_yyyymm')),
                'color': (row.get('color') or '').strip() or None,
                'mileage_km': parse_optional_int(row.get('mileage_km')),
            }
        )
        log_stage_progress(stage_name, index, len(rows))

    log_stage_footer(stage_name, started_at, output_rows=len(built_rows))
    return built_rows


async def run_string_to_bmt_stage(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    stage_name = 'string2BMT'
    started_at = log_stage_header(stage_name, len(rows))
    unique_titles = list(dict.fromkeys(str(row['title']) for row in rows))
    duplicate_count = len(rows) - len(unique_titles)
    titles_to_map = [title for title in unique_titles if title not in bmt_cache]
    reused_title_count = len(unique_titles) - len(titles_to_map)
    print(
        f'[{stage_name}] batch setup: unique_titles={len(unique_titles):,}, '
        f'duplicates={duplicate_count:,}, reused_titles={reused_title_count:,}, '
        f'titles_to_map={len(titles_to_map):,}, batch_size={BMT_BATCH_SIZE}, '
        f'max_concurrency={BMT_MAX_CONCURRENCY}, max_retries={BMT_MAX_RETRIES}'
    )

    mapped_title_count = reused_title_count
    mapped_success_count = 0
    for title in unique_titles:
        cached_brand, cached_model_name, _ = bmt_cache.get(title, (None, None, None))
        if cached_brand and cached_model_name:
            mapped_success_count += 1

    for batch_start in range(0, len(titles_to_map), BMT_BATCH_SIZE):
        title_batch = titles_to_map[batch_start:batch_start + BMT_BATCH_SIZE]
        batch_results = [
            match.as_tuple()
            for match in await STRING_TO_BMT_MAPPER.map_many_async(
                title_batch,
                use_agent=USE_AGENT,
                max_concurrency=BMT_MAX_CONCURRENCY,
                max_retries=BMT_MAX_RETRIES,
                retry_backoff_seconds=BMT_RETRY_BACKOFF_SECONDS,
            )
        ]

        for title, result in zip(title_batch, batch_results, strict=True):
            bmt_cache[title] = result
            if result[0] and result[1]:
                mapped_success_count += 1

        mapped_title_count += len(title_batch)
        log_stage_progress(
            stage_name,
            mapped_title_count,
            len(unique_titles),
            extra=f'mapped_titles={mapped_title_count:,}, successful_titles={mapped_success_count:,}',
        )

    mapped_rows: list[dict[str, object]] = []
    row_success_count = 0

    for index, row in enumerate(rows, start=1):
        brand, model_name, trim_name = bmt_cache.get(str(row['title']), (None, None, None))
        if brand and model_name:
            row_success_count += 1

        mapped_row = dict(row)
        mapped_row.update(
            {
                'brand': brand,
                'model_name': model_name,
                'trim_name': trim_name,
            }
        )
        mapped_rows.append(mapped_row)

        log_stage_progress(
            f'{stage_name}-attach',
            index,
            len(rows),
            extra=f'row_success={row_success_count:,}',
        )

    log_stage_footer(
        stage_name,
        started_at,
        output_rows=len(mapped_rows),
        unique_titles=len(unique_titles),
        duplicate_titles=duplicate_count,
        reused_titles=reused_title_count,
        successful_titles=mapped_success_count,
        successful_rows=row_success_count,
    )
    return mapped_rows


def run_category_stage(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    stage_name = 'BMTaddCAT'
    started_at = log_stage_header(stage_name, len(rows))
    categorized_rows: list[dict[str, object]] = []
    success_count = 0
    cache_hit_count = 0

    for index, row in enumerate(rows, start=1):
        key = (row.get('brand'), row.get('model_name'), row.get('trim_name'))
        if key in category_cache:
            cache_hit_count += 1
        else:
            category_cache[key] = get_major_category_from_tuple(*key)

        major_category = category_cache[key]
        if major_category is not None:
            success_count += 1

        categorized_row = dict(row)
        categorized_row['major_category'] = major_category
        categorized_rows.append(categorized_row)

        log_stage_progress(
            stage_name,
            index,
            len(rows),
            extra=f'success={success_count:,}, cache_hits={cache_hit_count:,}',
        )

    log_stage_footer(
        stage_name,
        started_at,
        output_rows=len(categorized_rows),
        success=success_count,
        cache_hits=cache_hit_count,
        unique_tuples=len(category_cache),
    )
    return categorized_rows


def run_score_stage(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    stage_name = 'BMT2Score'
    started_at = log_stage_header(stage_name, len(rows))
    scored_rows: list[dict[str, object]] = []
    success_count = 0
    cache_hit_count = 0

    for index, row in enumerate(rows, start=1):
        key = (row.get('brand'), row.get('model_name'), row.get('trim_name'))
        if key in score_cache:
            cache_hit_count += 1
        else:
            score_cache[key] = get_size_score(*key)

        size_score = score_cache[key]
        if size_score is not None:
            success_count += 1

        scored_row = dict(row)
        scored_row['size_score'] = round(size_score, 6) if size_score is not None else None
        scored_rows.append(scored_row)

        log_stage_progress(
            stage_name,
            index,
            len(rows),
            extra=f'success={success_count:,}, cache_hits={cache_hit_count:,}',
        )

    log_stage_footer(
        stage_name,
        started_at,
        output_rows=len(scored_rows),
        success=success_count,
        cache_hits=cache_hit_count,
        unique_tuples=len(score_cache),
    )
    return scored_rows


def finalize_training_ready_rows(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    stage_name = 'finalize'
    started_at = log_stage_header(stage_name, len(rows))
    finalized_rows: list[dict[str, object]] = []

    for index, row in enumerate(rows, start=1):
        finalized_rows.append({column: row.get(column) for column in OUTPUT_COLUMNS})
        log_stage_progress(stage_name, index, len(rows))

    log_stage_footer(stage_name, started_at, output_rows=len(finalized_rows))
    return finalized_rows


async def build_training_ready_rows_async() -> tuple[list[dict[str, object]], int]:
    try:
        base_rows = build_base_rows(sanitized_rows)
        bmt_rows = await run_string_to_bmt_stage(base_rows)
        category_rows = run_category_stage(bmt_rows)
        score_rows = run_score_stage(category_rows)
        training_ready_rows = finalize_training_ready_rows(score_rows)
        dropped_missing_size_score_count = sum(1 for row in training_ready_rows if row.get('size_score') is None)
        training_ready_rows = [row for row in training_ready_rows if row.get('size_score') is not None]
        return training_ready_rows, dropped_missing_size_score_count
    finally:
        await STRING_TO_BMT_MAPPER.aclose()


warm cache loaded: titles=851, categories=527, scores=527


## 5. 단계별 파이프라인 실행과 CSV 저장

아래 셀에서는 전체 row 목록을 stage 순서대로 통과시킵니다. 각 stage 는 자체 로그를 출력하고, `size_score` 가 없는 row 는 export 전에 전부 제거한 뒤 최종 CSV 로 저장합니다.


In [6]:
if 'ipykernel' in sys.modules:
    training_ready_rows, dropped_missing_size_score_count = await build_training_ready_rows_async()
else:
    import asyncio

    training_ready_rows, dropped_missing_size_score_count = asyncio.run(build_training_ready_rows_async())

with TRAINING_READY_CSV.open('w', encoding='utf-8-sig', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=OUTPUT_COLUMNS)
    writer.writeheader()
    writer.writerows(training_ready_rows)

print(f'dropped rows without size_score: {dropped_missing_size_score_count:,}')
print(f'training-ready row count: {len(training_ready_rows):,}')
print(f'training-ready csv saved to: {TRAINING_READY_CSV}')
print('\nfirst 3 training-ready rows:')
pprint(training_ready_rows[:3])


[base] start: total_rows=2,468
[base] progress: 26/2,468
[base] progress: 52/2,468
[base] progress: 78/2,468
[base] progress: 104/2,468
[base] progress: 130/2,468
[base] progress: 156/2,468
[base] progress: 182/2,468
[base] progress: 208/2,468
[base] progress: 234/2,468
[base] progress: 260/2,468
[base] progress: 286/2,468
[base] progress: 312/2,468
[base] progress: 338/2,468
[base] progress: 364/2,468
[base] progress: 390/2,468
[base] progress: 416/2,468
[base] progress: 442/2,468
[base] progress: 468/2,468
[base] progress: 494/2,468
[base] progress: 520/2,468
[base] progress: 546/2,468
[base] progress: 572/2,468
[base] progress: 598/2,468
[base] progress: 624/2,468
[base] progress: 650/2,468
[base] progress: 676/2,468
[base] progress: 702/2,468
[base] progress: 728/2,468
[base] progress: 754/2,468
[base] progress: 780/2,468
[base] progress: 806/2,468
[base] progress: 832/2,468
[base] progress: 858/2,468
[base] progress: 884/2,468
[base] progress: 910/2,468
[base] progress: 936/2,468


## 6. 결과 검증

최종 export 기준 row 수를 확인하고, 최종 테이블 안에 남아 있는 매핑 결과를 점검합니다.


In [7]:
mapping_success_count = sum(1 for row in training_ready_rows if row.get('brand') and row.get('model_name'))
category_success_count = sum(1 for row in training_ready_rows if row.get('major_category'))
score_success_count = sum(1 for row in training_ready_rows if row.get('size_score') is not None)

brand_counter = Counter(row['brand'] for row in training_ready_rows if row.get('brand'))
unmatched_rows = [
    row for row in training_ready_rows
    if not row.get('brand') or not row.get('model_name')
][:10]
uncategorized_rows = [row for row in training_ready_rows if row.get('brand') and row.get('model_name') and not row.get('major_category')][:10]
unscored_rows = [row for row in training_ready_rows if row.get('brand') and row.get('model_name') and row.get('size_score') is None][:10]

print(f'dropped rows without size_score: {dropped_missing_size_score_count:,}')
print(f'final exported row count: {len(training_ready_rows):,}')
print(f'mapping success count: {mapping_success_count:,}')
print(f'category success count: {category_success_count:,}')
print(f'score success count: {score_success_count:,}')
print('\nbrand distribution after mapping:')
pprint(brand_counter.most_common())
print('\nunmatched sample rows:')
pprint(unmatched_rows)
print('\nuncategorized sample rows:')
pprint(uncategorized_rows)
print('\nunscored sample rows:')
pprint(unscored_rows)


dropped rows without size_score: 337
final exported row count: 2,131
mapping success count: 2,131
category success count: 2,131
score success count: 2,131

brand distribution after mapping:
[('hyundai', 890),
 ('kia', 839),
 ('kgm', 175),
 ('chevrolet', 126),
 ('renault', 101)]

unmatched sample rows:
[]

uncategorized sample rows:
[]

unscored sample rows:
[]
